# 03_03 - Mapa SER con predicción proxy

Ejecución técnica del mapa SER+EMT con capa de facilidad proxy SER por barrio. El HTML se genera en `reports/maps/` y no se incrusta en el notebook.

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado data_catalog.csv.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/hugo/TFM_parking_madrid')

In [2]:
from src.models.ser_parking_proxy import build_ser_parking_proxy_from_paths
from src.visualization.parking_map import build_ser_prediction_map_from_operational

PATHS = {
    "m0_profiles": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_profiles.parquet",
    "m0_metadata": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_model_metadata.json",
    "capacidad_ser": ROOT / "data/processed/core/ser/ser_barrio_capacidad_anio.parquet",
    "autorizaciones": ROOT / "data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet",
    "ivtm_cero": ROOT / "data/interim/ser/ser_padron_vehiculos_ivtm_barrio/ser_padron_vehiculos_ivtm_barrio_clean.parquet",
    "calendario_laboral": ROOT / "data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet",
}

pd.DataFrame(
    [{"name": name, "path": path.relative_to(ROOT).as_posix(), "exists": path.exists()} for name, path in PATHS.items()]
)

,name,path,exists
0,m0_profiles,data/processed/core/ser/modeling/ser_m0_selected_profiles.parquet,True
1,m0_metadata,data/processed/core/ser/modeling/ser_m0_selected_model_metadata.json,True
2,capacidad_ser,data/processed/core/ser/ser_barrio_capacidad_anio.parquet,True
3,autorizaciones,data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet,True
4,ivtm_cero,data/interim/ser/ser_padron_vehiculos_ivtm_barrio/ser_padron_vehiculos_ivtm_barrio_clean.parquet,True
5,calendario_laboral,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,True


## Escenario

In [3]:
proxy_result = build_ser_parking_proxy_from_paths(
    m0_profiles_path=PATHS["m0_profiles"],
    m0_metadata_path=PATHS["m0_metadata"],
    capacidad_ser_path=PATHS["capacidad_ser"],
    autorizaciones_path=PATHS["autorizaciones"],
    ivtm_cero_path=PATHS["ivtm_cero"],
    calendario_laboral_path=PATHS["calendario_laboral"],
    scenario_datetime="2026-07-14 14:23:00",
    ivtm_reference_year=2025,
    output_mode="operational",
    expected_n_barrios=65,
    strict=True,
)

pd.DataFrame([proxy_result.metadata])

,model_id,target_column,training_period_start,training_period_end,scenario_datetime_requested,scenario_datetime_used,interval_alignment,interval_assignment,intervalo_inicio,intervalo_fin,intervalo_ajustado_30min,target_year,ivtm_reference_year,expected_n_barrios,dia_semana_num_m0,dia_semana_num_calendario,tipo_dia_calendario,ventana_ser_validada,weights,interpretation
0,M0_historical_profile,ocupacion_pagada_proxy,2023-01-02 09:00:00,2026-03-31 20:30:00,2026-07-14 14:23:00,2026-07-14 14:00:00,floor,contained_in_30min_interval,2026-07-14 14:00:00,2026-07-14 14:30:00,True,2026,2025,65,1,2,laborable,09:00-21:00,"{'w_m0': 0.6, 'w_estructura': 0.4, 'w_residentes_in_estructura': 0.75, 'w_cero_in_estructura': 0.25}",prob_aparcar_proxy es una escala proxy relativa; no es una probabilidad real observada de encontrar plaza.


## Mapa

In [4]:
map_result = build_ser_prediction_map_from_operational(
    operational=proxy_result.operational,
    scenario_metadata=proxy_result.metadata,
    root=ROOT,
    target_year=2026,
    html_output_path=Path("reports/maps/mapa_ser_emt_prediccion_proxy.html"),
    png_output_path=Path("reports/figures/ser_prediccion_proxy/mapa_ser_prediccion_proxy_ejemplo.png"),
)

map_result.outputs

,output,path,exists,size_mb
0,html_prediccion_proxy,reports/maps/mapa_ser_emt_prediccion_proxy.html,True,19.583
1,png_prediccion_proxy,reports/figures/ser_prediccion_proxy/mapa_ser_prediccion_proxy_ejemplo.png,True,3.070


## Checks

In [5]:
map_result.checks

,check_id,status,detail,critical
0,ser_layers_exist,OK,ser_geoportal_limite_ser=data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet:True; ser_geoportal_barrios_ser=data/interim/cartografia/ser_g...,True
1,ser_layers_not_empty,OK,ser_geoportal_limite_ser:rows=1; ser_geoportal_barrios_ser:rows=66; ser_geoportal_bandas_aparcamiento:rows=34450; callejero_viales_vigentes:rows=9287; ser_parquimetros:rows=4772,True
2,ser_crs_epsg_25830,OK,ser_geoportal_limite_ser:epsg=25830; ser_geoportal_barrios_ser:epsg=25830; ser_geoportal_bandas_aparcamiento:epsg=25830; callejero_viales_vigentes:epsg=25830; ser_parquimetros:...,True
3,emt_file_exists,OK,/Users/hugo/TFM_parking_madrid/data/processed/core/emt/inventario_global_emt.parquet,True
4,emt_not_empty,OK,rows=85,True
5,emt_expected_entities,OK,rows=85; expected=85,True
6,emt_required_columns,OK,missing=[],True
7,emt_parking_uid_not_null,OK,nulls=0,True
8,emt_parking_uid_unique,OK,duplicates=0,True
9,emt_coordinates_valid,OK,"lat_nulls=0; lon_nulls=0; lat_range=[40.36521668890186, 40.4941929509737]; lon_range=[-3.78414489510692, -3.598831]",True


## Diagnóstico de join

In [6]:
map_result.diagnostics["prediction_join"]

,n_barrios_mapa,n_barrios_operational,n_barrios_join,missing_after_join,map_not_prediction,prediction_not_map
0,65,65,65,0,,


## Categorías

In [7]:
map_result.diagnostics["prediction_categories"]

,categoria_prob_aparcar,categoria_prob_aparcar_label,n_barrios
0,baja,Baja,0
1,media,Media,49
2,alta,Alta,16


## Distribución

In [8]:
map_result.diagnostics["prediction_distribution"]

,n,min,p25,mean,p50,p75,max
0,65,0.456439,0.546077,0.625309,0.625278,0.691966,0.855478


## Escenario del mapa

In [9]:
map_result.diagnostics["scenario"]

,fecha,fecha_label,hora_solicitada,intervalo_inicio,intervalo_fin,intervalo_label,dia_semana_num,dia_semana,nota
0,2026-07-14,14/07/2026,14:23,2026-07-14 14:00:00,2026-07-14 14:30:00,14:00–14:30,1,martes,"Escala proxy relativa, no probabilidad observada de encontrar plaza."
